# FIGARO parser walkthrough

This notebook is the practical guide for parsing FIGARO from the Eurostat API in MARIO.

## What this notebook covers

- which Eurostat dataflows MARIO uses;
- how `SUT` and `IOT` parsing differ;
- how `year=`, `countries=`, and `iot_mode=` control the request;
- why no local FIGARO path is needed anymore.

## Source links

- Eurostat Statistics API: https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-introduction
- FIGARO data browser folder: https://ec.europa.eu/eurostat/databrowser/explore/all/naio?lang=en&subtheme=naio.naio_10.naio_10_fcp&display=list&sort=category

MARIO downloads FIGARO through the Eurostat API. The old local flat-file `path` workflow is deprecated.

## Dataflow groups

The parser selects the dataflow suffix from `year`:

- `2010`-`2013`: `s1`, `u1`, `ip1`, `ii1`
- `2014`-`2017`: `s2`, `u2`, `ip2`, `ii2`
- `2018`-`2021`: `s3`, `u3`, `ip3`, `ii3`
- `2022` onwards: `s4`, `u4`, `ip4`, `ii4`

For example, a 2022 SUT uses `naio_10_fcp_s4` and `naio_10_fcp_u4`.

## Key arguments

- `table`: choose `"SUT"` or `"IOT"`.
- `year`: required.
- `countries`: optional list of FIGARO country codes. Use it while developing to avoid downloading the full table.
- `iot_mode`: only for `table="IOT"`; use `"auto"`, `"product"`, or `"industry"`.
- `unit`: currently `"MIO_EUR"`.

## API notes and caveats

FIGARO is fully API-based. The old `path` argument is deprecated and ignored.

MARIO chunks requests by `c_orig` to avoid Eurostat extraction-size limits and then rebuilds the MARIO matrices locally. For `SUT`, `DOM` rows in the use table are interpreted as value-added rows. For `IOT`, `DOM` rows are interpreted in the same way for the `V` matrix.

Full FIGARO payloads are large, so `countries=` is the practical way to keep exploratory runs small.

In [1]:
import mario

## Parse a FIGARO SUT

The example below limits the request full FIGARO table

In [2]:
db_sut = mario.parse_figaro(
    table="SUT",
    year=2022,
    # path="/path/to/figaro/sut_directory",
    path='/Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-PolitecnicodiMilano/DENG-SESAM - Documenti/c-Research/a-Datasets/_Input Output Databases/FIGARO/SUT'
)

db_sut

INFO Parser: reading cached FIGARO API payload from /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-PolitecnicodiMilano/DENG-SESAM - Documenti/c-Research/a-Datasets/_Input Output Databases/FIGARO/SUT/naio_10_fcp_s4_2022_MIO_EUR_8ae0a3260d8f.csv.
INFO Parser: reading cached FIGARO API payload from /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-PolitecnicodiMilano/DENG-SESAM - Documenti/c-Research/a-Datasets/_Input Output Databases/FIGARO/SUT/naio_10_fcp_u4_2022_MIO_EUR_8be2fc37e8a5.csv.
INFO Parser: FIGARO SUT parsed from Eurostat API with 50 regions, 64 activities, 64 commodities, 6 factor rows.
INFO Metadata: initialized.


name = FIGARO SUT 2022
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 64
Commodity = 64
Factor of production = 6
Satellite account = 1
Consumption category = 5
Region = 50

In [3]:
db_sut

name = FIGARO SUT 2022
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 64
Commodity = 64
Factor of production = 6
Satellite account = 1
Consumption category = 5
Region = 50

## Parse a FIGARO IOT

For `IOT` parsing, `iot_mode="auto"` defaults to the product-by-product dataflow.

In [ ]:
from importlib.resources import path


db_iot = mario.parse_figaro(
    table="IOT",
    year=2022,
    iot_mode="product",
    path="/path/to/figaro/iot_directory",
)

db_iot

## First inspection

Once parsed, the result is a standard MARIO database.

In [4]:
db_sut.get_index("Region")[:5]

['Belgium', 'Bulgaria', 'Czechia', 'Denmark', 'Germany']

In [5]:
db_sut.get_index("Activity")[:5]

['Crop and animal production, hunting and related service activities',
 'Forestry and logging',
 'Fishing and aquaculture',
 'Mining and quarrying',
 'Manufacture of food products; beverages and tobacco products']

In [6]:
db_sut.get_index("Commodity")[:5]

['Products of agriculture, hunting and related services',
 'Products of forestry, logging and related services',
 'Fish and other fishing products; aquaculture products; support services to fishing',
 'Mining and quarrying',
 'Food, beverages and tobacco products']

## Practical warnings

- Full FIGARO tables are large to be downloaded; use `countries=` while developing workflows.
- MARIO chunks Eurostat API requests by `c_orig` internally to avoid extraction-size limits.
- The old local `path` argument is deprecated and ignored.